# Modelos de Clasificación 

En este proyecto buscamos clasificar tipos celulares a partir de representaciones reducidas mediante PCA. Dado que el problema es de clasificación multiclase con alto número de observaciones y el desbalance entre clases, se seleccionan modelos capaces de manejar datos de alta dimensionalidad reducida, relaciones no lineales y eficiencia computacional.
---
## Logistic Regression (Multinomial)

La regresión logística es un modelo lineal que estima la probabilidad de pertenencia a cada clase mediante una combinación lineal de las variables de entrada. En problemas multiclase utiliza la función softmax, que generaliza la regresión logística binaria para múltiples categorías. El modelo aprende un conjunto de coeficientes que ponderan cada componente del PCA. La predicción final se obtiene asignando la clase con mayor probabilidad estimada.

Matemáticamente, el modelo busca optimizar una función de pérdida basada en máxima verosimilitud, ajustando los pesos mediante descenso por gradiente.

### Parámetros 
- C: controla la regularización (menor valor = mayor regularización)
- solver: algoritmo de optimización (lbfgs, saga)
- class_weight: manejo de desbalance de clases
- max_iter: número de iteraciones del optimizador

### Utilidad
Es un modelo base fundamental porque:
- Funciona bien con datos ya reducidos por PCA
- Es rápido y escalable para 300k células
- Permite interpretar la importancia global de los componentes

---

## Linear Support Vector Machine (Linear SVM)

Linear SVM busca encontrar un hiperplano óptimo que separe las clases maximizando el margen entre ellas. En lugar de modelar probabilidades, se enfoca en la frontera de decisión. El algoritmo identifica los vectores de soporte (muestras críticas) que definen la separación entre clases. En espacios de alta dimensión como PCA, suele funcionar muy bien.

La optimización se basa en minimizar una función de pérdida hinge con regularización.

### Parámetros
- C: controla el margen de separación vs errores
- loss: función de pérdida (hinge o squared hinge)
- class_weight: ajuste por desbalance

### Utilidad
- Excelente desempeño en espacios reducidos (PCA)
- Muy eficiente en datasets grandes
- Buen equilibrio entre rendimiento y velocidad

---

## Random Forest

Random Forest es un modelo de ensamble basado en múltiples árboles de decisión. Cada árbol se entrena con una muestra aleatoria del dataset (bootstrap) y con un subconjunto aleatorio de variables. Cada árbol realiza particiones del espacio de características mediante reglas condicionales. La predicción final se obtiene por votación mayoritaria de todos los árboles.

Este enfoque reduce la varianza del modelo y mejora la generalización.

### Parámetros 
- n_estimators: número de árboles
- max_depth: profundidad máxima de los árboles
- min_samples_split: mínimo de muestras para dividir un nodo
- class_weight: balance de clases

### Utilidad
- Captura relaciones no lineales
- Robusto frente a ruido biológico
- Permite interpretar importancia de variables PCA

---

## XGBoost (Extreme Gradient Boosting)

XGBoost es un modelo de boosting basado en árboles de decisión. A diferencia de Random Forest, los árboles se construyen de forma secuencial, donde cada nuevo árbol corrige los errores del modelo anterior. El modelo optimiza una función de pérdida mediante gradiente descendente, incorporando regularización para evitar sobreajuste.

Cada iteración mejora la predicción global reduciendo el error residual.

### Parámetros 
- n_estimators: número de árboles secuenciales
- learning_rate: velocidad de aprendizaje
- max_depth: profundidad de cada árbol
- subsample: proporción de datos por árbol

### Utilidad 
- Alto rendimiento en clasificación multiclase
- Maneja relaciones complejas entre componentes PCA
- Excelente precisión en datasets biológicos

---

## Extra Trees Classifier

Extra Trees (Extremely Randomized Trees) es un ensamble de árboles de decisión similar a Random Forest, pero introduce mayor aleatoriedad. En lugar de buscar el mejor punto de división, selecciona puntos de corte aleatorios para cada variable, lo que reduce la varianza del modelo.

La predicción final se realiza por votación de todos los árboles.

### Parámetros 
- n_estimators: número de árboles
- max_features: número de variables consideradas en cada split
- max_depth: profundidad máxima
- bootstrap: uso o no de muestreo con reemplazo

### Utilidad 
- Más rápido que Random Forest
- Muy eficiente con PCA reducido
- Buen desempeño en datasets grandes y ruidosos

---

# Comparación de Modelos

| Modelo | Descripción | Parámetros clave |
|--------|-------------|------------------|
| Logistic Regression | Modelo lineal probabilístico multiclase basado en softmax | C, solver, class_weight, max_iter |
| Linear SVM | Clasificador lineal basado en margen óptimo entre clases | C, loss, class_weight |
| Random Forest | Ensamble de árboles con bagging y votación | n_estimators, max_depth, min_samples_split, class_weight |
| XGBoost | Boosting secuencial de árboles optimizando errores | n_estimators, learning_rate, max_depth, subsample |
| Extra Trees | Ensamble de árboles con divisiones aleatorias | n_estimators, max_features, max_depth, bootstrap |

In [9]:
import numpy as np
from pathlib import Path

from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

# Ruta de datos procesados
data_path = Path("..") / "data" / "processed"

# Cargar datasets completos
X_train = np.load(data_path / "X_train.npy")
X_test  = np.load(data_path / "X_test.npy")
y_train = np.load(data_path / "y_train.npy")
y_test  = np.load(data_path / "y_test.npy")

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (262536, 50) (262536,)
Test: (65634, 50) (65634,)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Modelo corregido
log_reg = LogisticRegression(
    solver="lbfgs",
    C=1.0,
    class_weight="balanced",
    max_iter=500,
    random_state=42
)

# Entrenamiento
log_reg.fit(X_train, y_train)

# Predicción
y_pred_lr = log_reg.predict(X_test)

# Evaluación
print("=== Logistic Regression ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr), 4))
print("F1-score (macro):", round(f1_score(y_test, y_pred_lr, average="macro"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

=== Logistic Regression ===
Accuracy: 0.9795
F1-score (macro): 0.9583

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.96      0.98     31528
           1       1.00      1.00      1.00     21332
           2       0.96      0.99      0.97      8285
           3       0.75      0.98      0.85      2291
           4       0.98      1.00      0.99      1816
           5       0.92      0.99      0.96       382

    accuracy                           0.98     65634
   macro avg       0.93      0.99      0.96     65634
weighted avg       0.98      0.98      0.98     65634



### Resultados: Logistic Regression

El modelo de regresión logística fue entrenado para clasificar tipos celulares a partir de componentes principales (PCA). A continuación se analiza su desempeño en el conjunto de prueba.

#### Métricas globales

- **Accuracy:** 0.9795  
- **F1-score (macro):** 0.9583  

 El accuracy (97.95%) indica que el modelo clasifica correctamente casi todas las células del conjunto de prueba. Y el F1-score macro (95.83%) es más relevante en este caso, ya que promedia el desempeño entre clases sin considerar su tamaño. Esto es importante porque el dataset está desbalanceado.

####  Análisis por clase

| Clase | Precisión | Recall | F1-score | Interpretación |
|------|-----------|--------|----------|----------------|
| 0 | 1.00 | 0.96 | 0.98 | Muy buen desempeño general, leve pérdida en recall |
| 1 | 1.00 | 1.00 | 1.00 | Clasificación perfecta en esta clase |
| 2 | 0.96 | 0.99 | 0.97 | Muy buen balance entre precisión y recall |
| 3 | 0.75 | 0.98 | 0.85 | Clase más difícil: muchos falsos positivos |
| 4 | 0.98 | 1.00 | 0.99 | Excelente desempeño |
| 5 | 0.92 | 0.99 | 0.96 | Buen rendimiento con ligera pérdida de precisión |


Buen desempeño general
El modelo logra un rendimiento alto gracias a:
- Uso de PCA (reducción de ruido y dimensionalidad)
- Separabilidad lineal razonable entre tipos celulares
- Uso de class_weight="balanced" que ayuda con el desbalance

#### Conclusiones

En general, la regresión logística se comporta como un modelo baseline muy sólido para este problema de clasificación de tipos celulares.

Se obtiene un rendimiento global alto, con una accuracy cercana al 98%, lo que indica que el espacio generado por PCA permite una separación bastante clara entre la mayoría de las clases. Esto sugiere que la reducción de dimensionalidad fue efectiva para eliminar ruido y conservar la estructura relevante de los datos biológicos.

Sin embargo, el análisis por clase muestra que el rendimiento no es completamente uniforme. La clase 3 presenta la principal dificultad, con una precisión menor, lo que indica confusión con otras clases. Este comportamiento es esperable en datos biológicos, donde ciertos tipos celulares pueden compartir perfiles de expresión similares y generar solapamientos en el espacio latente.

La comparación entre métricas macro y ponderadas confirma este fenómeno. Mientras el F1-score ponderado es alto debido al peso de las clases mayoritarias, el F1-score macro es ligeramente menor, evidenciando que el rendimiento disminuye en clases menos representadas o más complejas.

En síntesis, el modelo demuestra que:
- El problema es en gran parte linealmente separable en el espacio PCA.
- Se evidencia el desbalance de dificultad entre clases.
- Los modelos lineales son adecuados como punto de partida, pero no capturan completamente las clases más complejas.

Por lo tanto, tiene sentido continuar con modelos más robustos y no lineales como Random Forest o XGBoost, que podrían mejorar el desempeño en las clases más difíciles sin depender exclusivamente de separabilidad lineal.

In [11]:
from sklearn.svm import LinearSVC

# Modelo SVM lineal con configuración completa
svm_model = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual=True,
    C=1.0,
    class_weight="balanced",
    max_iter=3000,
    random_state=42
)

# Entrenamiento
svm_model.fit(X_train, y_train)

# Predicción
y_pred_svm = svm_model.predict(X_test)

# Evaluación
print("=== Linear SVM ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_svm), 4))
print("F1-score (macro):", round(f1_score(y_test, y_pred_svm, average="macro"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

=== Linear SVM ===
Accuracy: 0.9838
F1-score (macro): 0.968

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.98     31528
           1       1.00      1.00      1.00     21332
           2       0.97      0.98      0.97      8285
           3       0.83      0.96      0.89      2291
           4       0.99      1.00      0.99      1816
           5       0.95      0.99      0.97       382

    accuracy                           0.98     65634
   macro avg       0.95      0.98      0.97     65634
weighted avg       0.98      0.98      0.98     65634



c:\Users\Alumno\Downloads\analisis_scRNA\scRNA_env\Lib\site-packages\sklearn\svm\_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


### Resultados: Linear Support Vector Machine (SVM)

El modelo de Support Vector Machine con kernel lineal fue entrenado para clasificar tipos celulares a partir de componentes principales (PCA). A continuación se analiza su desempeño en el conjunto de prueba.

#### Métricas globales

- Accuracy: 0.9838  
- F1-score (macro): 0.968  

El accuracy (98.38%) indica que el modelo clasifica correctamente la gran mayoría de las muestras del conjunto de prueba. El F1-score macro (96.8%) es relevante porque promedia el desempeño entre clases sin estar influenciado por el desbalance del dataset.

#### Análisis por clase

| Clase | Precisión | Recall | F1-score | Interpretación |
|------|-----------|--------|----------|----------------|
| 0 | 0.99 | 0.98 | 0.98 | Muy buen desempeño general, con ligera mejora en estabilidad |
| 1 | 1.00 | 1.00 | 1.00 | Clasificación perfecta en esta clase |
| 2 | 0.97 | 0.98 | 0.97 | Desempeño muy sólido y equilibrado |
| 3 | 0.83 | 0.96 | 0.89 | Clase más compleja, con menor precisión relativa |
| 4 | 0.99 | 1.00 | 0.99 | Excelente rendimiento en esta clase |
| 5 | 0.95 | 0.99 | 0.97 | Muy buen desempeño general con alta consistencia |

#### Buen desempeño general

El modelo muestra un rendimiento alto y estable en la mayoría de las clases, lo que indica una buena capacidad de separación en el espacio de características generado por PCA.

Se observa que:

- La mayoría de las clases presentan valores altos de precisión y recall
- El modelo mantiene un comportamiento consistente incluso en clases minoritarias
- La clase 3 sigue siendo la más difícil de clasificar, aunque con un desempeño aceptable

#### Conclusiones

El modelo Linear SVM logra un desempeño global sólido en la tarea de clasificación de tipos celulares.

Se obtiene una accuracy cercana al 98%, lo que indica que la mayoría de las predicciones son correctas. El F1-score macro confirma un buen equilibrio general entre clases, lo que sugiere que el modelo no está fuertemente sesgado hacia las clases mayoritarias.

En el análisis por clase se observa que:

- Algunas clases son más fáciles de separar en el espacio PCA (especialmente las clases 1 y 4)
- La clase 3 presenta mayor dificultad, lo que indica solapamiento en el espacio de características
- El modelo mantiene buen rendimiento incluso en clases con menor representación

En síntesis, el modelo demuestra que el problema es mayoritariamente linealmente separable en el espacio reducido por PCA, aunque existen ciertas clases con mayor complejidad biológica que presentan mayores niveles de confusión.

In [12]:
from sklearn.ensemble import RandomForestClassifier

# Modelo Random Forest con configuración completa
rf_model = RandomForestClassifier(
    n_estimators=300,
    criterion="gini",
    max_depth=25,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    bootstrap=True,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Entrenamiento
rf_model.fit(X_train, y_train)

# Predicción
y_pred_rf = rf_model.predict(X_test)

# Evaluación
print("=== Random Forest ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))
print("F1-score (macro):", round(f1_score(y_test, y_pred_rf, average="macro"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

=== Random Forest ===
Accuracy: 0.9854
F1-score (macro): 0.9696

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.99     31528
           1       1.00      1.00      1.00     21332
           2       0.97      0.99      0.98      8285
           3       0.86      0.94      0.90      2291
           4       0.99      0.99      0.99      1816
           5       1.00      0.94      0.97       382

    accuracy                           0.99     65634
   macro avg       0.97      0.97      0.97     65634
weighted avg       0.99      0.99      0.99     65634



### Resultados: Random Forest

El modelo Random Forest fue entrenado para clasificar tipos celulares a partir de componentes principales (PCA). A continuación se analiza su desempeño en el conjunto de prueba.

#### Métricas globales

- Accuracy: 0.9854  
- F1-score (macro): 0.9696  

El accuracy (98.54%) indica que el modelo clasifica correctamente la gran mayoría de las células del conjunto de prueba. El F1-score macro (96.96%) es relevante porque evalúa el desempeño promedio entre clases sin verse influenciado por el desbalance del dataset.

#### Análisis por clase

| Clase | Precisión | Recall | F1-score | Interpretación |
|------|-----------|--------|----------|----------------|
| 0 | 0.99 | 0.98 | 0.99 | Muy buen desempeño general, alta estabilidad |
| 1 | 1.00 | 1.00 | 1.00 | Clasificación perfecta en esta clase |
| 2 | 0.97 | 0.99 | 0.98 | Muy buen balance entre precisión y recall |
| 3 | 0.86 | 0.94 | 0.90 | Clase más compleja, con mayor dificultad de separación |
| 4 | 0.99 | 0.99 | 0.99 | Excelente desempeño, muy consistente |
| 5 | 1.00 | 0.94 | 0.97 | Alta precisión, con ligera pérdida en recall |

#### Buen desempeño general

El modelo muestra un rendimiento alto y consistente en la mayoría de las clases, lo que indica que el enfoque basado en árboles es capaz de capturar relaciones no lineales presentes en el espacio de características.

Se observa que:
- Las clases mayoritarias presentan desempeño muy estable y cercano a perfecto
- Las clases minoritarias mantienen buenos niveles de precisión y recall
- La clase 3 sigue siendo la más desafiante, aunque con mejor desempeño relativo en comparación con otras configuraciones

#### Conclusiones

El modelo Random Forest logra un desempeño global muy alto en la tarea de clasificación de tipos celulares.

Se obtiene una accuracy cercana al 98.5%, lo que indica que la mayoría de las predicciones son correctas. El F1-score macro confirma un buen equilibrio general entre clases, lo que sugiere que el modelo no está fuertemente sesgado hacia las clases mayoritarias.

En el análisis por clase se observa que:

- El modelo mantiene un rendimiento muy alto en la mayoría de las clases
- La clase 3 presenta mayor dificultad, lo que sugiere solapamiento en el espacio de características biológicas
- Incluso las clases minoritarias mantienen un desempeño estable, lo que indica buena capacidad de generalización

En síntesis, el modelo demuestra que el problema contiene relaciones no lineales que pueden ser capturadas eficientemente por métodos basados en árboles, permitiendo una clasificación robusta incluso en un contexto de desbalance y alta dimensionalidad reducida mediante PCA.

In [14]:
from xgboost import XGBClassifier

# Modelo XGBoost con configuración completa
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    gamma=0.1,
    min_child_weight=3,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=len(set(y_train)),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# Entrenamiento
xgb_model.fit(X_train, y_train)

# Predicción
y_pred_xgb = xgb_model.predict(X_test)

# Evaluación
print("=== XGBoost ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_xgb), 4))
print("F1-score (macro):", round(f1_score(y_test, y_pred_xgb, average="macro"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xgb))

=== XGBoost ===
Accuracy: 0.9895
F1-score (macro): 0.9773

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     31528
           1       1.00      1.00      1.00     21332
           2       0.98      0.98      0.98      8285
           3       0.92      0.92      0.92      2291
           4       1.00      0.99      1.00      1816
           5       0.99      0.95      0.97       382

    accuracy                           0.99     65634
   macro avg       0.98      0.97      0.98     65634
weighted avg       0.99      0.99      0.99     65634



### Resultados: XGBoost

El modelo XGBoost fue entrenado para clasificar tipos celulares a partir de componentes principales (PCA). A continuación se analiza su desempeño en el conjunto de prueba.

#### Métricas globales

- Accuracy: 0.9895  
- F1-score (macro): 0.9773  

El accuracy (98.95%) indica que el modelo clasifica correctamente la gran mayoría de las muestras del conjunto de prueba. El F1-score macro (97.73%) es relevante porque evalúa el desempeño promedio entre clases sin verse influenciado por el desbalance del dataset.

#### Análisis por clase

| Clase | Precisión | Recall | F1-score | Interpretación |
|------|-----------|--------|----------|----------------|
| 0 | 0.99 | 0.99 | 0.99 | Excelente desempeño, muy estable |
| 1 | 1.00 | 1.00 | 1.00 | Clasificación perfecta en esta clase |
| 2 | 0.98 | 0.98 | 0.98 | Muy buen equilibrio entre métricas |
| 3 | 0.92 | 0.92 | 0.92 | Clase más compleja, buen desempeño pero con mayor dificultad relativa |
| 4 | 1.00 | 0.99 | 1.00 | Excelente desempeño general |
| 5 | 0.99 | 0.95 | 0.97 | Muy buen rendimiento, con ligera pérdida en recall |

#### Buen desempeño general

El modelo muestra un rendimiento muy alto y consistente en todas las clases, lo que indica una fuerte capacidad de capturar relaciones no lineales complejas en el espacio de características generado por PCA.

Se observa que:

- Las clases mayoritarias presentan un desempeño prácticamente perfecto
- Las clases minoritarias mantienen un rendimiento alto y estable
- La clase 3 es la más desafiante, aunque con un desempeño sólido comparado con el resto

#### Conclusiones

El modelo XGBoost logra un rendimiento global muy alto en la tarea de clasificación de tipos celulares.

Se obtiene una accuracy cercana al 99%, lo que indica que la mayoría de las predicciones son correctas. El F1-score macro confirma un equilibrio muy alto entre clases, lo que sugiere que el modelo maneja adecuadamente tanto clases mayoritarias como minoritarias.

En el análisis por clase se observa que:

- El modelo mantiene un rendimiento consistente en todas las clases
- La clase 3 presenta mayor dificultad relativa, aunque con métricas aún altas
- Las demás clases presentan resultados cercanos a óptimos, lo que indica una buena capacidad de generalización

En síntesis, el modelo demuestra una alta capacidad para capturar patrones complejos en los datos, beneficiándose de su naturaleza de boosting, que combina múltiples árboles débiles para construir un modelo altamente preciso y robusto frente a desbalance y relaciones no lineales.

In [15]:
from sklearn.ensemble import ExtraTreesClassifier

# Modelo Extra Trees con configuración completa
et_model = ExtraTreesClassifier(
    n_estimators=400,
    criterion="gini",
    max_depth=30,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    bootstrap=False,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Entrenamiento
et_model.fit(X_train, y_train)

# Predicción
y_pred_et = et_model.predict(X_test)

# Evaluación
print("=== Extra Trees ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_et), 4))
print("F1-score (macro):", round(f1_score(y_test, y_pred_et, average="macro"), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_et))

=== Extra Trees ===
Accuracy: 0.9835
F1-score (macro): 0.9673

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.98      0.98     31528
           1       1.00      1.00      1.00     21332
           2       0.97      0.99      0.98      8285
           3       0.81      0.96      0.88      2291
           4       0.99      1.00      1.00      1816
           5       1.00      0.95      0.97       382

    accuracy                           0.98     65634
   macro avg       0.96      0.98      0.97     65634
weighted avg       0.98      0.98      0.98     65634

